# Notebook 03 — Prompt engineering avanzato

**Modulo:** 03
**Tempo stimato:** 2-2,5 ore

## Cosa costruirai

Al termine di questo notebook avrai:
- Un confronto diretto Chain of Thought attivato vs disattivato, sullo stesso problema
- Un confronto diretto pattern persona attivato vs disattivato
- Un template con placeholder riutilizzabile
- Un mini root prompt che rifiuta input fuori formato
- Una pipeline finale che combina tutti i pattern avanzati del Modulo 03

## Prerequisiti

```bash
pip install anthropic python-dotenv
```

Serve il file `.env` con `ANTHROPIC_API_KEY` (lo stesso dei moduli precedenti).

---
## Cella 0 — Setup

Riusiamo lo stesso schema del Modulo 02: una funzione `chiama_llm` che
richiama il client Anthropic senza doverlo reinizializzare a ogni cella.

In [ ]:
import os
from dotenv import load_dotenv
import anthropic

load_dotenv()
ANTHROPIC_KEY = os.getenv("ANTHROPIC_API_KEY")

if not ANTHROPIC_KEY:
    print("MANCA ANTHROPIC_API_KEY. Controlla il tuo file .env.")
else:
    print(f"OK  ANTHROPIC_API_KEY: {ANTHROPIC_KEY[:8]}...")

cliente = anthropic.Anthropic(api_key=ANTHROPIC_KEY)
MODELLO = "claude-haiku-4-5-20251001"  # verifica l'ID modello aggiornato nella doc ufficiale

def chiama_llm(prompt: str, system: str = "Sei un assistente utile e preciso.", max_tokens: int = 600) -> str:
    risposta = cliente.messages.create(
        model=MODELLO,
        max_tokens=max_tokens,
        system=system,
        messages=[{"role": "user", "content": prompt}]
    )
    return risposta.content[0].text

print("Funzione chiama_llm pronta.")


---
## Cella 1 — Chain of Thought: prima e dopo

Sottoponiamo lo stesso problema di calcolo in due versioni: una che chiede
direttamente il risultato, una che forza il modello a ragionare passo per
passo prima di rispondere.

**Prima di eseguire:** secondo te, su un problema come questo, con un
modello "veloce" come Haiku la differenza sarà evidente o marginale?

In [ ]:
problema = """
Un'azienda vende un prodotto a 45 euro. Il costo di produzione è 28 euro
per unità. Il mese scorso ha venduto 340 unità, ma questo mese prevede un
aumento del 15% nelle vendite grazie a una nuova campagna, con un costo
fisso aggiuntivo di 800 euro per la campagna stessa.
Qual è il margine di profitto netto atteso per questo mese?
"""

prompt_diretto = f"""
{problema}
Rispondi solo con la cifra finale in euro.
"""

prompt_cot = f"""
{problema}

Risolvi seguendo questi passaggi, mostrando il calcolo di ciascuno prima di
passare al successivo:
1. Calcola il margine unitario (prezzo - costo di produzione).
2. Calcola le unità previste per questo mese (vendite del mese scorso + 15%).
3. Calcola il margine lordo totale (margine unitario × unità previste).
4. Sottrai il costo fisso della campagna per ottenere il margine netto.
5. Presenta il risultato finale chiaramente etichettato.
"""

print("=== SENZA Chain of Thought ===")
print(chiama_llm(prompt_diretto, max_tokens=100))
print()
print("=== CON Chain of Thought ===")
print(chiama_llm(prompt_cot, max_tokens=500))


**Osservazioni:** verifica tu stesso il calcolo a mano (unità previste =
340 × 1,15 = 391; margine unitario = 45 - 28 = 17; margine lordo = 391 ×
17; margine netto = margine lordo - 800). La versione senza Chain of
Thought è più veloce ma più a rischio di errore aritmetico "nascosto" in un
unico salto; la versione con Chain of Thought è più lenta (più token
generati, quindi più costosa) ma il ragionamento è verificabile passo per
passo. Annota qui il risultato dei due esperimenti e se coincidono con il
calcolo a mano:

- Risultato senza CoT: ...
- Risultato con CoT: ...
- Coincidono con il calcolo a mano? ...

---
## Cella 2 — Pattern persona: prima e dopo

Stessa domanda, con e senza un ruolo assegnato esplicitamente al modello.

In [ ]:
domanda_tecnica = "Valuta se conviene usare un database relazionale o un database a documenti per un'applicazione di e-commerce con 50.000 prodotti e forte necessità di transazioni consistenti sugli ordini."

prompt_senza_persona = domanda_tecnica

prompt_con_persona = f"""
Agisci come un architetto software con 15 anni di esperienza in sistemi
di e-commerce ad alto traffico, specializzato in scelte di persistenza dei
dati. Rispondi con il rigore tecnico che useresti in un design document
interno, citando trade-off concreti.

{domanda_tecnica}
"""

print("=== SENZA pattern persona ===")
print(chiama_llm(prompt_senza_persona, max_tokens=300))
print()
print("=== CON pattern persona ===")
print(chiama_llm(prompt_con_persona, max_tokens=300))


**Osservazioni:** confronta il livello di dettaglio tecnico, l'uso di
terminologia di settore (ACID, sharding, consistenza forte/eventuale...) e
la struttura della risposta. Nella maggior parte dei casi la versione con
persona è più concreta e meno "da manuale introduttivo".

---
## Cella 3 — Esercizio: template con placeholder

**TODO:** costruisci una funzione `genera_proposta` che accetta come
parametri `nome_azienda`, `settore` e `nome_prodotto`, li sostituisce in un
template con placeholder, e restituisce il testo generato dal modello.
Testala con almeno due combinazioni diverse di parametri.

In [ ]:
TEMPLATE_PROPOSTA = """
Scrivi una breve proposta commerciale (massimo 120 parole) per
<NOME_AZIENDA>, che opera nel settore <SETTORE>, per il prodotto
<NOME_PRODOTTO>. Tono professionale, diretto, senza frasi fatte.
"""

def genera_proposta(nome_azienda: str, settore: str, nome_prodotto: str) -> str:
    # TODO: sostituisci i placeholder nel template e chiama chiama_llm
    prompt = TEMPLATE_PROPOSTA  # <-- modifica questa riga
    return chiama_llm(prompt, max_tokens=250)


# print(genera_proposta("Verdenova Srl", "agricoltura di precisione", "sensori IoT per l'irrigazione"))
# print(genera_proposta("Fintrack SpA", "contabilità per PMI", "software di riconciliazione automatica"))


---
## Cella 4 — Root prompt: rifiutare input fuori formato

Costruiamo un mini sistema che accetta **solo** richieste di percorso nel
formato esatto:

```
Origine: <località>
Destino: <località>
```

Qualunque altro input, anche se la richiesta è chiara per un umano, deve
ricevere un messaggio di errore fisso — non un tentativo "gentile" di
interpretarla comunque.

In [ ]:
ROOT_PROMPT = """
Rispondi SOLO se il messaggio dell'utente segue esattamente questo formato,
su due righe:
Origine: <località>
Destino: <località>

Se il formato non è rispettato ESATTAMENTE (comprese le due righe con
queste etichette), non tentare in nessun caso di interpretare l'intenzione
dell'utente, anche se ti sembra chiara. Rispondi esattamente con questo
testo, senza aggiungere altro:
"Formato non valido. Usa: Origine: <località> / Destino: <località>"

Se il formato è rispettato, rispondi con una frase che conferma origine e
destino, nulla di più.
"""

richieste_test = [
    "Origine: Milano\nDestino: Roma",             # formato corretto
    "Vorrei andare da Milano a Roma",              # formato NON rispettato, ma chiaro per un umano
    "Origine: Milano",                             # formato incompleto
]

for richiesta in richieste_test:
    print(f"INPUT:\n{richiesta}")
    print(f"OUTPUT:\n{chiama_llm(richiesta, system=ROOT_PROMPT, max_tokens=100)}")
    print("-" * 50)


**Cosa osservare:** la seconda richiesta è perfettamente comprensibile
per un umano ("Milano → Roma"), ma il root prompt impone al modello di
rifiutarla comunque perché non rispetta il formato. Questo è esattamente il
comportamento voluto in molti sistemi aziendali: prevedibilità del formato
sopra la flessibilità dell'interpretazione. Se il modello ha comunque
provato a "essere utile" interpretando la richiesta libera, prova a
rafforzare ulteriormente l'istruzione (per esempio ripetendo il divieto in
modo ancora più esplicito) e osserva come cambia il comportamento.

---
## Sfida finale — Assistente finanziario con persona, CoT e root prompt

Costruisci un unico prompt di sistema che combina tre pattern di questo
modulo:

1. **Root prompt:** l'assistente accetta solo domande che iniziano con
   `Domanda finanziaria:` — qualunque altro input riceve un messaggio di
   errore fisso.
2. **Pattern persona:** quando il formato è rispettato, l'assistente
   risponde nei panni di un consulente finanziario indipendente.
3. **Chain of Thought:** per domande che richiedono un calcolo, l'assistente
   deve mostrare i passaggi del ragionamento prima della conclusione finale.

Testa il tuo sistema con almeno tre input: uno fuori formato, uno nel
formato corretto ma senza calcoli, uno nel formato corretto con un calcolo
da fare.

In [ ]:
SYSTEM_ASSISTENTE_FINANZIARIO = """
# TODO: scrivi qui il system prompt che combina root prompt + persona + CoT
"""

domande_test = [
    "Quanto rende un ETF azionario?",                                          # fuori formato
    "Domanda finanziaria: cos'è la diversificazione di portafoglio?",          # formato ok, no calcolo
    "Domanda finanziaria: se investo 5000 euro al 4% annuo composto per 3 anni, quanto avrò alla fine?",  # formato ok, con calcolo
]

# for domanda in domande_test:
#     print(f"INPUT: {domanda}")
#     print(chiama_llm(domanda, system=SYSTEM_ASSISTENTE_FINANZIARIO, max_tokens=400))
#     print("-" * 50)


---
## Autovalutazione della settimana

1. **Cosa ha funzionato bene questa settimana?**
   ...

2. **Cosa non hai capito del tutto?**
   ...

3. **Cosa cambieresti nel tuo approccio la settimana prossima?**
   ...

4. **Tempo reale dedicato:**
   ...

---
*Notebook 03 — Modulo 03 del Corso Completo di IA Generativa*